<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# Introduction to Regularization

_Authors: Matt Brems + Tim Book_

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')

## Learning Objectives

1. Describe what a loss function is.
2. Define regularization.
3. Describe and differentiate LASSO and Ridge regularization.
4. Understand how regularization affects the bias-variance tradeoff.
5. Implement LASSO regression and Ridge regression.

## Review

<details><summary>What is the bias-variance tradeoff?</summary>

- Mean squared error can be decomposed into a bias component plus a variance component (plus a systematic error, but we don't have control over this part, so we often ignore it).
- The bias-variance tradeoff refers to the fact that taking steps to minimize bias usually comes at the expense of an increase in variance. Similarly, taking steps to minimize variance usually comes at the expense of an increase in bias.

</details>

---

<details><summary>What evidence/information would lead me to believe that my model suffers from high variance?</summary>
    
- After splitting my data into training and testing sets, if I see that my model performs way better on my training set than my testing set, this means that my model is not generalizing very well to "new" data.
- An example might be where our training MSE is substantially lower than our testing MSE, or where our training R-squared is substantially higher than our testing R-squared.
</details>

---

<details><summary>What term do we use to describe a model that suffers from high variance?</summary>

- Overfit.
</details>

## Why is high variance bad?

High variance is bad because it means that our model doesn't generalize well to new data. This means that our model looks as though it performs well on our training data but won't perform as well on new, unseen data.

---
<details><summary>How might we try to fix a model that suffers from high variance?</summary>

- Gather more data. (Although this is usually expensive and time-consuming.)
- Drop features.
- Make our existing features less complex. (i.e. get rid of interaction terms or higher order terms.)
- Choose a simpler model.
- Regularization!
</details>

## Pop Math Quiz

### Problem 1
**What is the value of $b$ that minimizes...**

$$ (y - b)^2 $$

<details><summary></summary>
When $b = y$, this expression has value 0. Since it's squared, it can't go below that.
</details>

### Problem 2
**What is the value of $b$ that minimizes...**

$$ (y - b)^2 + \alpha b^2 $$

where $\alpha > 0$?

<details><summary></summary>
This is more complicated, isn't it? You can use calculus and come up with an answer:
    
$$ \hat{b} = \frac{y}{1 + \alpha} $$

But what is the effect of $\alpha$ on our solution?
</details>

## Overview of regularization

---

"Regularizing" regression models is to:
- **automatically** avoid overfitting 
- while we fit our model
- by adding a "penalty" to our loss function.

Remember what our loss function has looked like so far:

$$
\text{minimize: }SSE = \sum_{i=1}^n (y_i - \hat{y}_i)^2 = \|\mathbf{y} - \hat{\mathbf{y}}\|^2
$$

In ordinary least squares (OLS) linear regression, we find the line of best fit by selecting the values of $\hat{\pmb\beta}$ that minimize the sum of squared errors (SSE).

In [ ]:
# Create figure.
plt.figure(figsize=(15, 7))

# Set up 200 steps between 0 and 8.
beta_1 = np.linspace(0, 8, 200)

# Create our SSE function. (This is a made up loss function!)
sse = beta_1**2 - 7*beta_1 + 13

# Set axis labels, title, and ranges.
plt.ylabel('Loss Function', fontsize=20)
plt.xlabel(r'Value of $\hat{\beta}_1$', fontsize=20) # The r allows us to typeset with LaTeX.
plt.title(r'What is our loss function for each $\hat{\beta}_1$ value?',
          fontsize=28,
          x=0,
          ha='left')
plt.xlim(0, 8)
plt.ylim(0, 30)

# Plot SSE for values of beta_1 between 0 and 8.
plt.plot(beta_1, sse, label='SSE', c='navy', linewidth=3)

plt.legend(fontsize=20);

<details><summary>Based on the above plot, what value for beta_1 should my model select?</summary>
    
- When fitting our model, we want to minimize our loss function.
- Our loss function is minimized at 3.5.
</details>

Regularization adds an extra piece to our loss function to avoid overfitting. When fitting a regularized linear regression model, we'll minimize this **new** loss function, which is the sum of squared errors plus a penalty.

$$
\begin{eqnarray*}
\text{minimize: }SSE + \text{Ridge penalty} &=& \sum_{i=1}^n (y_i - \mathbf{x}_i^T\beta)^2 + \alpha \sum_{j=1}^p \beta_j^2 \\
                                            &=& \|\mathbf{y} - \mathbf{X}\beta\|^2 + \alpha\|\beta\|^2
\end{eqnarray*}
$$

### NOTE: This is written a few ways!
In fact, most commonly in the literature, the **regularization** parameter is a $\lambda$ instead of an $\alpha$. Scikit-Learn writes $\alpha$, so we do too.

In [ ]:
# Create figure.
plt.figure(figsize=(15, 7))

# Set up 200 steps between 0 and 8.
beta_1 = np.linspace(0, 8, 200)

# Create our SSE function. (This is a made up loss function!)
sse = beta_1**2 - 7*beta_1 + 13

# Create our Regularized Loss function.
alpha = 1
reg_loss = sse + alpha*beta_1**2

# Set axis labels, title, and ranges.
plt.ylabel('Loss Function', fontsize=20)
plt.xlabel(r'Value of $\hat{\beta}_1$', fontsize=20) # The r allows us to typeset with LaTeX.
plt.title(r'What is our loss function for each $\hat{\beta}_1$ value?',
          fontsize=28,
          x=0,
          ha='left')
plt.xlim(0, 8)
plt.ylim(0, 75)

# Plot SSE for values of beta_1 between 0 and 8.
plt.plot(beta_1, sse, label='SSE', linewidth=3, color='navy')

# Plot Regularized Loss for values of beta_1 between 0 and 8.
plt.plot(beta_1, reg_loss, label=f'Regularized Loss with alpha = {alpha}', linewidth=3, color='orange')

# Plot legend.
plt.legend(fontsize=20);

<details><summary>If I used my regularized loss function to fit my model, what would my value for beta_1 be?</summary>
    
- Our regularized loss function is minimized around 1.8.
</details>

#### What does this do?
This new loss function ensures the coefficients don't get too big when they shouldn't!

Put another way, this new loss function makes sure our coefficients are based on signal (the thing we want to detect/understand) and not noise (the random fluctuations in our data we want to ignore).
- If the coefficients are large because of noise, then our loss function will likely gently slope, allowing regularization to push the coefficients considerably toward zero.
- If the coefficients are large because they're supposed to be large, then our loss function will likely be steep and regularization will have little effect on the coefficients.
    - To visually check this, try using `sse = 8*beta_1**2 - 10*beta_1 + 13` as your loss function. Look at how little of an effect regularization has on the value of beta_1!

The two most common types of regularization are the **Ridge** and **Lasso**. There is also mixture of them called the **Elastic Net**.

## What is the effect of regularization?

---

**To demonstrate the effects of regularization, we will be using a dataset on wine quality.**

### Load the wine .csv

This version has red and white wines concatenated together and tagged with a binary 1/0 indicator (1 is red wine). There are many other variables purportedly related to the rated quality of the wine.

In [ ]:
# Load in the wine .csv.
wine = pd.read_csv('datasets/winequality_merged.csv')

# Convert all columns to lowercase and replace spaces in column names.

In [ ]:
# Check the first five rows.

In [ ]:
# How big is this dataset?

In [ ]:
# Check for missing values.

### Create the target and predictor matrix 

Our goal today is to show you how regularization can prevent overfitting. We are going to create a very complex dataset that will overfit to our data.

#### With sklearn

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Create X and y.
X = wine.drop('quality', axis=1)
y = wine['quality']

# Instantiate our PolynomialFeatures object to create all two-way terms.
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)

# Fit and transform our X data.
X_overfit = poly.fit_transform(X)

In [ ]:
poly.get_feature_names(X.columns)

In [ ]:
# Check out the dimensions of X_overfit.

#### Let's split our data up into training and testing sets. Why do we split our data into training and testing sets?

In [ ]:
# Import train_test_split.
from sklearn.model_selection import train_test_split

In [ ]:
# Create train/test splits.
X_train, X_test, y_train, y_test = train_test_split(
    X_overfit,
    y,
    test_size=0.7,
    random_state=42
)

In [ ]:
print(f'X_train shape is: {X_train.shape}')
print(f'y_train shape is: {y_train.shape}')
print(f'X_test shape is: {X_test.shape}')
print(f'y_test shape is: {y_test.shape}')

<a id='standardization'></a>

## Standardizing predictors is required

---

When regularizing, **it is neccessary to standardize the predictor columns before constructing the models, even the dummy coded categorical variables**.

Below we define our target variable, then normalize the columns that are not the target.

We are going to change our loss function to add a penalty. The equations for the Ridge and Lasso penalties:

### $$ \text{Ridge penalty}\; = \alpha \sum_{j=1}^p \beta_j^2$$

### $$ \text{Lasso penalty}\; = \alpha \sum_{j=1}^p |\beta_j|$$

<details><summary>Why might I need to standardize my variables before regularizing?</summary>
    
- Regularization penalizes the magnitude (size) of the coefficients.
- The size of the coefficient depends on the scale of the independent (X) variable.
- Suppose I wanted to predict one's income using their high school GPA and SAT score. 
$$
\begin{eqnarray*}
\text{income} &=& \beta_0 + \beta_1\text{GPA} + \beta_2\text{SAT}
\end{eqnarray*}
$$
</details>

In [ ]:
# Import StandardScaler.
from sklearn.preprocessing import StandardScaler

In [ ]:
# Instantiate Standard Scaler.

# Fit AND transform Standard Scaler to overfit data in one step.

<a id='viz-ridge'></a>

## Defining and Visualizing the Ridge

---

### $$ \text{minimize:}\; SSE+Ridge = \sum_{i=1}^n \left(y_i - \mathbf{x}_i\beta\right)^2 + \alpha\sum_{j=1}^p \beta_j^2$$

**Where:**

$\beta_j^2$ is the squared coefficient for variable $x_j$.

$\sum_{j=1}^n \beta_j^2$ is the sum of these squared coefficients for every variable in the model. 
- This does **not** include the intercept $\beta_0$.

$\alpha$ is a constant for the _strength_ of the regularization parameter. The higher the value, the greater the impact of this new component in the loss function. If the value was zero, we would revert back to just the least squares loss function. If the value was a billion, however, the residual sum of squares component would have a much smaller effect on the loss/cost than the regularization term.

### Let's think about this...

$$ \|\mathbf{y} - \mathbf{X}\beta\|^2 + \alpha\|\beta\|^2 $$

<details><summary>What's the optimal value of $\beta$ when $\alpha = 0$?</summary>
Our problem reduces to OLS, so it's the good old fashioned OLS solution! For the math nerds playing along from home, that's:
    
$$ \hat{\beta} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y} $$
</details>

<details><summary>What's the optimal value of $\beta$ when $\alpha = \infty$?</summary>
Anything besides $\hat{\beta} = \mathbf{0}$ will cause our whole loss function to be $\infty$. So, it must be that $\hat{\beta} = \mathbf{0}$!
</details>

### We can look at a traceplot to see this:

![](imgs/ridge-trace.png)

In [ ]:
# Ridge regressor lives here:
from sklearn.linear_model import Ridge

<a id='viz-lasso'></a>

## Defining the LASSO

---

Now we do the same thing as above but for the LASSO.

LASSO regression takes a different approach. Instead of adding the sum of squared $\beta$ coefficients to the SSE, it adds the sum of the absolute values of the $\beta$ coefficients:

$$ \text{minimize:}\; SSE + \text{LASSO} = \sum_{i=1}^n \left(y_i - \mathbf{x}_i^T\beta\right)^2 + \alpha\sum_{j=1}^p |\beta_j|$$

Also written:

$$ \|\mathbf{y} - \mathbf{X}\beta \|^2 + \alpha\|\beta\|_1 $$

**Where:**

$|\beta_j|$ is the absolute value of the $\beta$ coefficient for variable $x_j$.

$\alpha$ is again the strength of the regularization penalty component in the loss function. 

### The LASSO traceplot looks a little different...
But I don't want to show it to you yet! We'll see it soon and discuss what LASSO actually does differently from Ridge.

In [ ]:
# LASSO regressor lives here:
from sklearn.linear_model import Lasso

<a id='model-performance'></a>

## Model performance of the Ridge and LASSO on the overfit data

### (i.e., what do I actually need to do?)

---

Let's check out how the penalties affect the performance of our models. Let's use the overfit data instead to see if regularization helps us control overfitting.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso

**Let's start with plain vanilla linear regression:**

In [ ]:
# Instantiate linear regression model without regularization.

In [ ]:
# Fit linear regression model.

In [ ]:
# Let's use R2 to see if our model is overfit, underfit, or neither.

<details><summary>How do we interpret this $R^2$ score?</summary>
22.0% of the variability in our $Y$ (wine quality) is explained by our model, relative to a baseline model with no predictors.
</details>

<details><summary>Given these two $R^2$ scores together, what do you conclude?</summary>
Because our training performance is better than our testing performance, it's likely that our model is overfit (and thus too complex). We should consider regularizing.
</details>

### Ridge Regression

**Now let's make it into a Ridge regression by adding the penalty term:**

In [ ]:
# Instantiate.

# Fit.

# Evaluate model using R2.

Which $\alpha$ to choose?  Let's search! (Remember that we search logarithmic space for Ridge regularization.)

In [ ]:
from sklearn.linear_model import RidgeCV

In [ ]:
# Set up a list of ridge alphas to check.
# Generates 100 values equally between 0 and 5,
# then converts them to alphas between 10^0 and 10^5.

# Cross-validate over our list of ridge alphas.

# Fit model using best ridge alpha!

In [ ]:
# Here is the optimal value of alpha

Our `ridge_model` object is actually already the model witht he optimal $\alpha$. Let's get the corresponding value of $R^2$.

### Lasso Regression

In [ ]:
from sklearn.linear_model import LassoCV

In [ ]:
# Reminder of OLS results:
lm = LinearRegression()
lm.fit(Z_train, y_train)
print(lm.score(Z_train, y_train))
print(lm.score(Z_test, y_test))

In [ ]:
# Set up a list of Lasso alphas to check.

# Cross-validate over our list of Lasso alphas.

# Fit model using best ridge alpha!

In [ ]:
# Here is the optimal value of alpha

### Let's check out the coefficients of the Lasso and Ridge models.

## Cliffsnotes: L.A.S.S.O.
LASSO is actually an acronym:

* **L**east
* **A**bsolute
* **S**hrinkage and
* **S**election
* **O**perator

**SHRINKAGE**: Higher $\alpha$ "shrinks" $\beta$ towards $\mathbf{0}$.

**SELECTION**: Higher $\alpha$ zeros out small $\beta$s.

![](imgs/lasso-trace.svg)

## So, um, what was LASSO doing here?
If you're an ultra math nerd, you might have noticed something fishy about our "penalty parameter" $\alpha$. We're doing an optimization problem, so actually, this $\alpha$ is a **Lagrange multiplier**. This means that optimizing our loss function:

$$ \|\mathbf{y} - \mathbf{X}\beta\|^2 + \alpha\|\beta\|_1 $$

is equivalent to optimizing the **constrained loss function**:

$$ \|\mathbf{y} - \mathbf{X}\beta\|^2 \quad \text{such that} \quad \|\beta\|_1 \le t $$

## [BRING IN THE APP!](https://timothykbook.shinyapps.io/RegularizationPlot/)

# Regularizing Logistic Regression: You've been doing it all along!

In [ ]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

### Let's take a look at the LogisticRegression documentation:

In [ ]:
LogisticRegression()

### Not only have you been regularizing this whole time, but you can't turn it off!
Unless regularization is necessary, **it should not be done!!** While you can't turn it completely off can set `C`, the inverse of `alpha`, extremely high. 

If you _do_ want to regularize, note that there is a much friendlier `LogisticRegressionCV` we will use.

In [ ]:
from sklearn.datasets import make_classification

In [ ]:
X, y = make_classification(
    n_samples=1000,
    n_features=200,
    n_informative=15,
    random_state=123
)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=123)

sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

In [ ]:
# Fit unregulatized logistic regression

# Overfit!

In [ ]:
# Fi
logreg_cv = LogisticRegressionCV(Cs=10, cv=5, penalty="l1", solver="liblinear")
logreg_cv.fit(X_train_sc, y_train)

print(logreg_cv.score(X_train_sc, y_train))
print(logreg_cv.score(X_test_sc, y_test))

In [ ]:
logreg_cv.C_

## Important Notes
- The $\alpha$ hyperparameter for regularization is **unrelated** to significance level in hypothesis testing.
- In certain resources, including [ISLR](http://www-bcf.usc.edu/~gareth/ISL/ISLR%20Seventh%20Printing.pdf), you'll see that $\lambda$ is used instead of $\alpha$ for regularization strength.
- We must standardize before regularizing, but regularization and standardization are not the same things!
- **FROM NOW ON, YOU MUST PAY ATTENTION TO REGULARIZATION WHEN CONDUCTING LOGISTIC REGRESSION!!!**

## Recap
- Regularization is used when evidence suggests our model is suffering from high error due to variance.
- Evidence to suggest our model suffers from high error due to variance includes substantially better performance on our training set than our testing set.
- LASSO tends to be "more brutal" than Ridge regularization in that it will zero out coefficients.
- If you want to combine LASSO and Ridge regularization, there is a technique called "ElasticNet" that does exactly this.

## ElasticNet Regression (bonus)

---

Can't decide?

![](imgs/why-not-both.jpg)

The Elastic Net combines the Ridge and Lasso penalties.  It adds *both* penalties to the loss function:

$$
\begin{eqnarray}
SSE + Ridge + Lasso &=& \sum_{i=1}^n \left(y_i - \hat{y}_i\right)^2 + \alpha\left[\rho\sum_{j=1}^p |\beta_j| + (1-\rho)\sum_{j=1}^p \beta_j^2\right] \\
&=& \|\mathbf{y} - \mathbf{X}\beta\|^2 + \alpha\left(\rho\|\beta\|_1 + (1 - \rho)\|\beta\|^2\right)
\end{eqnarray}
$$


In the elastic net, the effect of the ridge versus the lasso is balanced by the $\rho$ parameter.  It is the ratio of Lasso penalty to Ridge penalty and must be between zero and one.

`ElasticNet` in sklearn has two parameters:
- `alpha`: the regularization strength.
- `l1_ratio`: the amount of L1 vs L2 penalty (i.e., $\rho$). An l1_ratio of 0 is equivalent to the Ridge, whereas an l1_ratio of 1 is equivalent to the Lasso.


In [ ]:
from sklearn.linear_model import ElasticNet

Calculate the coefficients with both alpha values and an l1_ratio of 0.05. Lasso can "overpower" the Ridge penalty in some datasets, and so rather than an equal balance I'm just adding a little bit of Lasso in.
- Using a $\rho$ value below 0.05 can empirically cause issues in `sklearn`.

In [ ]:
from sklearn.linear_model import ElasticNetCV

In [ ]:
# Set up a list of alphas to check.
enet_alphas = np.linspace(0.5, 1.0, 100)

# Set up our l1 ratio. (What does this do?)
enet_ratio = 0.5

# Instantiate model.
enet_model = ElasticNetCV(alphas=enet_alphas, l1_ratio=enet_ratio, cv=5)

# Fit model using optimal alpha.
enet_model = enet_model.fit(X_train, y_train)

# Generate predictions.
enet_model_preds = enet_model.predict(X_test)
enet_model_preds_train = enet_model.predict(X_train)

# Evaluate model.
print(enet_model.score(X_train, y_train))
print(enet_model.score(X_test, y_test))

In [ ]:
# Here is the optimal value of alpha.
enet_model.alpha_